In [1]:
print("hello world")

hello world


In [2]:
from __future__ import annotations

import json
import os
import re
import subprocess
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import pandas as pd

In [14]:
WORK_DIR = Path(r"C:\dsarp_work")
OUTPUT_DIR = Path(r"C:\dsarp_outputs")

MAX_COMMITS_PER_REPO = 25




REPOSITORIES = [
    {"name": "tika", "url": "https://github.com/apache/tika.git"},
    {"name": "maven", "url": "https://github.com/apache/maven.git"},
    {"name": "camel", "url": "https://github.com/apache/camel.git"},
    {"name": "ant", "url": "https://github.com/apache/ant.git"},
    {"name": "lucene", "url": "https://github.com/apache/lucene.git"},
]

In [38]:
from pathlib import Path

REFACTORING_MINER = r"C:\RM\bin\RefactoringMiner.bat"

print(Path(REFACTORING_MINER).exists())

True


In [40]:
# verifying the refactoring miner
print(REFACTORING_MINER)

print(run([REFACTORING_MINER, "--help"], check=False))

C:\RM\bin\RefactoringMiner.bat



In [41]:
print(run([REFACTORING_MINER, "--help"], check=False))

In [32]:
# testing the refactoring miner
run([REFACTORING_MINER, "-h"], check=False)

''

In [4]:
import re
import json
import subprocess
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass

PACKAGE_RE = re.compile(r"^\s*package\s+([A-Za-z0-9_.]+)\s*;", re.M)
IMPORT_RE = re.compile(r"^\s*import\s+(?:static\s+)?([A-Za-z0-9_.]+)", re.M)
TYPE_RE = re.compile(r"\b(?:class|interface|enum|record)\s+[A-Za-z_][A-Za-z0-9_]*")

SMELL_PRIORS = {
    "Cyclic Dependency": {"Move Class", "Move Method", "Extract Class", "Extract Interface"},
    "Hub-like Dependency": {"Extract Class", "Move Method", "Extract Method", "Move Class"},
    "Unstable Dependency": {"Move Class", "Extract Interface", "Pull Up Method", "Push Down Method"},
    "Large Component": {"Extract Class", "Extract Method", "Move Method", "Move Attribute"},
    "Excessive Package Coupling": {"Move Class", "Extract Interface", "Extract Class", "Move Method"},
}

@dataclass
class Metrics:
    commit: str
    java_files: int
    packages: int
    package_edges: int
    cyclic_packages: int
    max_fan_in: int
    max_fan_out: int
    unstable_dependencies: int
    hub_like_packages: int
    large_packages: int
    median_package_size: float

In [18]:
import subprocess

def run(cmd, cwd=None, check=True):
    process = subprocess.run(
        cmd,
        cwd=cwd,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        encoding="utf-8",
        errors="replace",
    )

    if check and process.returncode != 0:
        raise RuntimeError(
            "Command failed:\n"
            f"cmd: {cmd}\n"
            f"cwd: {cwd}\n"
            f"returncode: {process.returncode}\n"
            f"stdout:\n{process.stdout}\n"
            f"stderr:\n{process.stderr}"
        )

    return process.stdout.strip()    


def checkout(repo, commit):
    repo = Path(repo)

    if not (repo / ".git").exists():
        raise RuntimeError(f"Not a Git repository: {repo.resolve()}")

    lock_file = repo / ".git" / "index.lock"
    if lock_file.exists():
        lock_file.unlink()

    run(["git", "reset", "--hard", "HEAD"], cwd=repo)
    run(["git", "clean", "-fd"], cwd=repo)
    run(["git", "checkout", "--force", "--quiet", commit], cwd=repo)

In [7]:
run(["git", "--version"], check=False)

'git version 2.42.0.windows.2'

In [13]:
run(["git", "config", "--global", "core.longpaths", "true"])
#enabling long paths for git

''

In [15]:
import shutil
import os
import stat


In [16]:
import shutil

def remove_readonly(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)


def clone_or_update(repos, repos_dir):
    repos_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    run(["git", "config", "--global", "core.longpaths", "true"])

    for repo in repos:
        name = repo["name"]
        url = repo["url"].strip()
        target = repos_dir / name

        if not url.startswith("https://github.com/") or not url.endswith(".git"):
            raise ValueError(f"Bad Git URL for {name}: {url}")

        if target.exists() and not (target / ".git").exists():
            print(f"Removing incomplete clone for {name}")
            shutil.rmtree(target, onerror=remove_readonly)

        if target.exists():
            print(f"Updating {name}")
            run(["git", "config", "core.longpaths", "true"], cwd=target)
            run(["git", "fetch", "--all", "--prune"], cwd=target)
            run(["git", "restore", "--source=HEAD", ":/"], cwd=target, check=False)
        else:
            print(f"Cloning {name} from {url}")
            try:
                run(["git", "clone", url, str(target)])
            except RuntimeError as exc:
                if target.exists() and (target / ".git").exists():
                    print(f"Clone partly succeeded for {name}; retrying checkout")
                    run(["git", "config", "core.longpaths", "true"], cwd=target)
                    run(["git", "restore", "--source=HEAD", ":/"], cwd=target)
                else:
                    raise exc

        commits = int(run(["git", "rev-list", "--count", "HEAD"], cwd=target))

        rows.append({
            "repository": name,
            "url": url,
            "path": str(target),
            "commits": commits,
        })

    return pd.DataFrame(rows)




In [17]:
repo_summary = clone_or_update(REPOSITORIES, WORK_DIR / "repositories")
repo_summary

Cloning tika from https://github.com/apache/tika.git
Cloning maven from https://github.com/apache/maven.git
Cloning camel from https://github.com/apache/camel.git
Cloning ant from https://github.com/apache/ant.git
Cloning lucene from https://github.com/apache/lucene.git


,repository,url,path,commits
0,tika,https://github.com/apache/tika.git,C:\dsarp_work\repositories\tika,10486
1,maven,https://github.com/apache/maven.git,C:\dsarp_work\repositories\maven,16165
2,camel,https://github.com/apache/camel.git,C:\dsarp_work\repositories\camel,81762
3,ant,https://github.com/apache/ant.git,C:\dsarp_work\repositories\ant,15118
4,lucene,https://github.com/apache/lucene.git,C:\dsarp_work\repositories\lucene,39324


In [19]:
def candidate_commits(repo, limit):
    terms = [
        "refactor",
        "extract",
        "move",
        "rename",
        "inline",
        "clean",
        "decompos",
        "modular",
        "architecture",
        "dependency",
    ]

    cmd = ["git", "log", "--date-order", "--reverse", "--regexp-ignore-case", "--format=%H"]

    for term in terms:
        cmd += ["--grep", term]

    commits = list(dict.fromkeys(run(cmd, cwd=repo, check=False).splitlines()))

    if limit is None or limit < 0:
        return commits

    return commits[:limit]


candidate_counts = []

for repo in REPOSITORIES:
    repo_path = WORK_DIR / "repositories" / repo["name"]
    candidate_counts.append({
        "repository": repo["name"],
        "candidate_commits": len(candidate_commits(repo_path, MAX_COMMITS_PER_REPO)),
    })

pd.DataFrame(candidate_counts)

,repository,candidate_commits
0,tika,25
1,maven,25
2,camel,25
3,ant,25
4,lucene,25


In [20]:
def commit_metadata(repo, commit):
    raw = run(["git", "show", "-s", "--format=%H%n%P%n%aI%n%s", commit], cwd=repo)
    lines = raw.splitlines()

    return {
        "commit": lines[0],
        "parents": lines[1].split() if len(lines) > 1 and lines[1] else [],
        "date": lines[2] if len(lines) > 2 else "",
        "subject": lines[3] if len(lines) > 3 else "",
    }


def get_refactorings(refminer, repo, commit, cache_file):
    cache_file.parent.mkdir(parents=True, exist_ok=True)

    if not cache_file.exists():
        run([refminer, "-c", str(repo), commit, "-json", str(cache_file)])

    data = json.loads(cache_file.read_text(encoding="utf-8"))
    commits = data.get("commits", [])

    if not commits:
        return []

    return commits[0].get("refactorings", [])


def refactoring_labels(refactorings):
    return sorted({
        item.get("type", "")
        for item in refactorings
        if item.get("type")
    })

In [21]:
def java_files(repo):
    ignored = {".git", "target", "build", ".gradle", "out", ".idea"}

    for path in repo.rglob("*.java"):
        if not any(part in ignored for part in path.parts):
            yield path


def package_of(source):
    match = PACKAGE_RE.search(source)
    return match.group(1) if match else "<default>"


def strongly_connected_components(graph):
    index = 0
    stack = []
    indices = {}
    lows = {}
    on_stack = set()
    components = []

    def visit(node):
        nonlocal index

        indices[node] = lows[node] = index
        index += 1
        stack.append(node)
        on_stack.add(node)

        for neighbor in graph.get(node, set()):
            if neighbor not in indices:
                visit(neighbor)
                lows[node] = min(lows[node], lows[neighbor])
            elif neighbor in on_stack:
                lows[node] = min(lows[node], indices[neighbor])

        if lows[node] == indices[node]:
            component = set()

            while True:
                item = stack.pop()
                on_stack.remove(item)
                component.add(item)

                if item == node:
                    break

            components.append(component)

    for node in graph:
        if node not in indices:
            visit(node)

    return components

In [22]:
def compute_metrics(repo, commit):
    checkout(repo, commit)

    package_sizes = Counter()
    package_graph = defaultdict(set)
    java_count = 0

    for file_path in java_files(repo):
        source = file_path.read_text(encoding="utf-8", errors="ignore")
        java_count += 1

        package = package_of(source)
        package_sizes[package] += max(1, len(TYPE_RE.findall(source)))
        package_graph.setdefault(package, set())

        for imported in IMPORT_RE.findall(source):
            parts = imported.split(".")

            if len(parts) >= 3:
                imported_package = ".".join(parts[:-1])

                if imported_package != package:
                    package_graph[package].add(imported_package)
                    package_graph.setdefault(imported_package, set())

    fan_out = {pkg: len(deps) for pkg, deps in package_graph.items()}
    fan_in = Counter(dep for deps in package_graph.values() for dep in deps)

    cycles = [
        component
        for component in strongly_connected_components(package_graph)
        if len(component) > 1
    ]

    sizes = sorted(package_sizes.values())
    median = float(sizes[len(sizes) // 2]) if sizes else 0.0
    large_threshold = max(10.0, median * 2.0)

    unstable = 0
    hubs = 0
    all_packages = set(package_graph) | set(fan_in)

    for package in all_packages:
        incoming = fan_in.get(package, 0)
        outgoing = fan_out.get(package, 0)
        total = incoming + outgoing
        instability = outgoing / total if total else 0.0

        if incoming >= 5 and instability > 0.8:
            unstable += 1

        if incoming >= 8 and outgoing >= 8:
            hubs += 1

    return Metrics(
        commit=commit,
        java_files=java_count,
        packages=len(package_sizes),
        package_edges=sum(fan_out.values()),
        cyclic_packages=sum(len(component) for component in cycles),
        max_fan_in=max(fan_in.values()) if fan_in else 0,
        max_fan_out=max(fan_out.values()) if fan_out else 0,
        unstable_dependencies=unstable,
        hub_like_packages=hubs,
        large_packages=sum(
            1 for value in package_sizes.values()
            if value >= large_threshold
        ),
        median_package_size=median,
    )

In [23]:
def infer_smells(before, after):
    checks = [
        ("Cyclic Dependency", "cyclic_packages", 3.0),
        ("Hub-like Dependency", "hub_like_packages", 2.5),
        ("Unstable Dependency", "unstable_dependencies", 2.0),
        ("Large Component", "large_packages", 1.5),
        ("Excessive Package Coupling", "package_edges", 1.0),
    ]

    rows = []

    for smell, metric, weight in checks:
        before_value = getattr(before, metric)
        after_value = getattr(after, metric)
        delta = before_value - after_value

        if before_value > 0:
            rows.append({
                "smell": smell,
                "metric": metric,
                "before": before_value,
                "after": after_value,
                "delta": delta,
                "improved": delta > 0,
                "score": max(0.0, delta * weight),
            })

    return rows


def rank_labels(smell_row, labels):
    ranked = []
    priors = SMELL_PRIORS.get(smell_row["smell"], set())

    for label in labels:
        score = smell_row["score"]

        if label in priors:
            score += 2.0

        if not smell_row["improved"]:
            score *= 0.25

        ranked.append((label, round(score, 4)))

    return [
        label
        for label, _ in sorted(ranked, key=lambda item: (-item[1], item[0]))
    ]

In [25]:
# lets mine one repository
def mine_repository(repo_config, max_commits_per_repo=25):
    repo_name = repo_config["name"]
    repo_path = WORK_DIR / "repositories" / repo_name
    cache_dir = WORK_DIR / "refactoringminer" / repo_name

    original_head = run(["git", "rev-parse", "HEAD"], cwd=repo_path)
    commits = candidate_commits(repo_path, max_commits_per_repo)

    dataset_rows = []
    metric_rows = []
    ranking_rows = []
    error_rows = []

    print(f"{repo_name}: {len(commits)} candidate commits")

    try:
        for index, commit in enumerate(commits, start=1):
            print(f"{repo_name} {index}/{len(commits)} {commit[:12]}")

            try:
                meta = commit_metadata(repo_path, commit)

                if len(meta["parents"]) != 1:
                    continue

                parent = meta["parents"][0]

                checkout(repo_path, original_head)

                refactorings = get_refactorings(
                    REFACTORING_MINER,
                    repo_path,
                    commit,
                    cache_dir / f"{commit}.json",
                )

                labels = refactoring_labels(refactorings)

                if not labels:
                    continue

                before = compute_metrics(repo_path, parent)
                after = compute_metrics(repo_path, commit)

                metric_rows.append({
                    "repository": repo_name,
                    "version": "before",
                    **asdict(before),
                })

                metric_rows.append({
                    "repository": repo_name,
                    "version": "after",
                    **asdict(after),
                })

                for smell in infer_smells(before, after):
                    ranked = rank_labels(smell, labels)

                    for position, label in enumerate(ranked, start=1):
                        ranking_rows.append({
                            "repository": repo_name,
                            "commit": commit,
                            "architecture_smell": smell["smell"],
                            "refactoring_type": label,
                            "rank": position,
                        })

                    dataset_rows.append({
                        "repository": repo_name,
                        "commit": commit,
                        "parent_commit": parent,
                        "author_date": meta["date"],
                        "subject": meta["subject"],
                        "architecture_smell": smell["smell"],
                        "metric_name": smell["metric"],
                        "metric_before": smell["before"],
                        "metric_after": smell["after"],
                        "metric_delta": smell["delta"],
                        "improved": smell["improved"],
                        "refactoring_labels": "|".join(labels),
                        "ranked_refactoring_labels": "|".join(ranked),
                        "top_refactoring_label": ranked[0] if ranked else "",
                        "input_text": (
                            f"Architecture smell: {smell['smell']}. "
                            f"Metric {smell['metric']} changed from "
                            f"{smell['before']} to {smell['after']}. "
                            f"Commit message: {meta['subject']}"
                        ),
                        "target_text": " | ".join(ranked),
                    })

            except Exception as exc:
                error_rows.append({
                    "repository": repo_name,
                    "commit": commit,
                    "error": str(exc)[:1000],
                })

                try:
                    checkout(repo_path, original_head)
                except Exception as cleanup_exc:
                    error_rows.append({
                        "repository": repo_name,
                        "commit": commit,
                        "error": f"cleanup_failed: {str(cleanup_exc)[:1000]}",
                    })

    finally:
        checkout(repo_path, original_head)

    return (
        pd.DataFrame(dataset_rows),
        pd.DataFrame(metric_rows),
        pd.DataFrame(ranking_rows),
        pd.DataFrame(error_rows),
    )

In [44]:
sample_dataset, sample_metrics, sample_ranking, sample_errors = mine_repository(
    REPOSITORIES[0],
    max_commits_per_repo=25,
)

sample_dataset.head()

tika: 25 candidate commits
tika 1/25 6e3ee160366e
tika 2/25 2b86daf8815f
tika 3/25 d363b828bc6e
tika 4/25 eb9d2e97dd2a
tika 5/25 1e2373c719f2
tika 6/25 53f61c8270e2
tika 7/25 f8183f2dffaa
tika 8/25 83cb301f046c
tika 9/25 d3e678bf2c00
tika 10/25 7bdb1c8dea70
tika 11/25 a03498cc8930
tika 12/25 53d14c5d024a
tika 13/25 9a00212a62c0
tika 14/25 f00e6fb40fea
tika 15/25 62e58ea2e8cc
tika 16/25 aceff84889da
tika 17/25 d064cb2f7ffd
tika 18/25 3fb58b7b5dc6
tika 19/25 e759bbbc8afc
tika 20/25 9fce256813d1
tika 21/25 b0a87ad5afe0
tika 22/25 e1da9a1fc19a
tika 23/25 580824e1088a
tika 24/25 f7079fdafc6d
tika 25/25 a8d1e675a2c1


,repository,commit,parent_commit,author_date,subject,architecture_smell,metric_name,metric_before,metric_after,metric_delta,improved,refactoring_labels,ranked_refactoring_labels,top_refactoring_label,input_text,target_text
0,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Hub-like Dependency,hub_like_packages,1,1,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
1,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Large Component,large_packages,2,2,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Large Component. Metric large_packages changed from 2 to 2. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
2,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Excessive Package Coupling,package_edges,133,131,2,True,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Excessive Package Coupling. Metric package_edges changed from 133 to 131. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
3,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Cyclic Dependency,cyclic_packages,2,2,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Cyclic Dependency. Metric cyclic_packages changed from 2 to 2. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class
4,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Hub-like Dependency,hub_like_packages,1,1,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class


In [45]:
sample_dataset.shape, sample_metrics.shape, sample_ranking.shape, sample_errors.shape

((59, 16), (30, 13), (271, 5), (0, 0))

In [29]:
sample_dataset.head(20)

""


In [36]:
pd.set_option("display.max_colwidth",2000)
sample_errors.head(20)


,repository,commit,error
0,tika,6e3ee160366ebd6b336d7b9f12984486a3bb886e,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', '6e3ee160366ebd6b336d7b9f12984486a3bb886e', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\6e3ee160366ebd6b336d7b9f12984486a3bb886e.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
1,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', '2b86daf8815f3c8d974f887e4cc064b9159db595', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\2b86daf8815f3c8d974f887e4cc064b9159db595.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
2,tika,d363b828bc6e714aa5f4ffedfbd1d09e1880f9ee,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', 'd363b828bc6e714aa5f4ffedfbd1d09e1880f9ee', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\d363b828bc6e714aa5f4ffedfbd1d09e1880f9ee.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
3,tika,eb9d2e97dd2a84b6275d5ccc87236e1cad88738c,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', 'eb9d2e97dd2a84b6275d5ccc87236e1cad88738c', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\eb9d2e97dd2a84b6275d5ccc87236e1cad88738c.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
4,tika,1e2373c719f24de4c92ec075350cfbe3c3838a71,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', '1e2373c719f24de4c92ec075350cfbe3c3838a71', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\1e2373c719f24de4c92ec075350cfbe3c3838a71.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
5,tika,53f61c8270e23cba731fa7186ba597fcc1f454d6,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', '53f61c8270e23cba731fa7186ba597fcc1f454d6', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\53f61c8270e23cba731fa7186ba597fcc1f454d6.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
6,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', 'f8183f2dffaac516ea48fc3522c302fa4937b26c', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\f8183f2dffaac516ea48fc3522c302fa4937b26c.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
7,tika,83cb301f046cf5621e6e71bcd907fef52c491078,"Command failed:\ncmd: ['C:\\Users\\bobby\\Downloads\\RefactoringMiner-3.1.4\\RefactoringMiner-3.1.4\\bin\\RefactoringMiner.bat', '-c', 'C:\\dsarp_work\\repositories\\tika', '83cb301f046cf5621e6e71bcd907fef52c491078', '-json', 'C:\\dsarp_work\\refactoringminer\\tika\\83cb301f046cf5621e6e71bcd907fef52c491078.json']\ncwd: None\nreturncode: 255\nstdout:\n\nstderr:\nThe input line is too long.\nThe syntax of the command is incorrect.\n"
8,tika,d3e678bf2c00961a33854ad3a0ecb0df7dd23b08,"Command failed:\ncmd: ['C:\\Users\\bobby\\Down

In [30]:
pd.set_option("display.max_colwidth", 1000)
sample_errors.head(25)

,repository,commit,error
0,tika,6e3ee160366ebd6b336d7b9f12984486a3bb886e,[WinError 2] The system cannot find the file specified
1,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,[WinError 2] The system cannot find the file specified
2,tika,d363b828bc6e714aa5f4ffedfbd1d09e1880f9ee,[WinError 2] The system cannot find the file specified
3,tika,eb9d2e97dd2a84b6275d5ccc87236e1cad88738c,[WinError 2] The system cannot find the file specified
4,tika,1e2373c719f24de4c92ec075350cfbe3c3838a71,[WinError 2] The system cannot find the file specified
5,tika,53f61c8270e23cba731fa7186ba597fcc1f454d6,[WinError 2] The system cannot find the file specified
6,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,[WinError 2] The system cannot find the file specified
7,tika,83cb301f046cf5621e6e71bcd907fef52c491078,[WinError 2] The system cannot find the file specified
8,tika,d3e678bf2c00961a33854ad3a0ecb0df7dd23b08,[WinError 2] The system cannot find the file specified
9,tika,7bdb1c8dea70014b3528736755868c526dfcf834,[WinError 2] The system cannot find the file specified


In [43]:
# checking if the refactorings commits dont exist in the sample dataset because the first 25 commits simply not useful, since this has finally worked meaning the refactoring miner works we can run the sample dataset and then also check the shape of it
# so there was nothing wrong with the first 25 commits, we just need to fix the path for the refactoring miner tool which was set too long
repo_path = WORK_DIR / "repositories" / "tika"
test_commit = candidate_commits(repo_path, 25)[0]
test_cache = WORK_DIR / "refactoringminer_test.json"

print("commit:", test_commit)
print("repo:", repo_path.resolve())
print("refminer:", REFACTORING_MINER)

if test_cache.exists():
    test_cache.unlink()

run([
    REFACTORING_MINER,
    "-c",
    str(repo_path.resolve()),
    test_commit,
    "-json",
    str(test_cache.resolve()),
])

print("json exists:", test_cache.exists())
print(test_cache.read_text(encoding="utf-8")[:2000])

commit: 6e3ee160366ebd6b336d7b9f12984486a3bb886e
repo: C:\dsarp_work\repositories\tika
refminer: C:\RM\bin\RefactoringMiner.bat
json exists: True
{
"commits": [
{
	"repository": "https://github.com/apache/tika.git",
	"sha1": "6e3ee160366ebd6b336d7b9f12984486a3bb886e",
	"url": "https://github.com/apache/tika/commit/6e3ee160366ebd6b336d7b9f12984486a3bb886e",
	"refactorings": []
}]
}


In [46]:
# Mining from every repository, but still the max_commits per repo is 25
all_datasets = []
all_metrics = []
all_rankings = []
all_errors = []

for repo in REPOSITORIES:
    dataset_part, metrics_part, ranking_part, errors_part = mine_repository(
        repo,
        max_commits_per_repo=MAX_COMMITS_PER_REPO,
    )

    all_datasets.append(dataset_part)
    all_metrics.append(metrics_part)
    all_rankings.append(ranking_part)
    all_errors.append(errors_part)

dataset = pd.concat(all_datasets, ignore_index=True) if all_datasets else pd.DataFrame()
metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
ranking = pd.concat(all_rankings, ignore_index=True) if all_rankings else pd.DataFrame()
errors = pd.concat(all_errors, ignore_index=True) if all_errors else pd.DataFrame()

dataset.shape, metrics.shape, ranking.shape, errors.shape

tika: 25 candidate commits
tika 1/25 6e3ee160366e
tika 2/25 2b86daf8815f
tika 3/25 d363b828bc6e
tika 4/25 eb9d2e97dd2a
tika 5/25 1e2373c719f2
tika 6/25 53f61c8270e2
tika 7/25 f8183f2dffaa
tika 8/25 83cb301f046c
tika 9/25 d3e678bf2c00
tika 10/25 7bdb1c8dea70
tika 11/25 a03498cc8930
tika 12/25 53d14c5d024a
tika 13/25 9a00212a62c0
tika 14/25 f00e6fb40fea
tika 15/25 62e58ea2e8cc
tika 16/25 aceff84889da
tika 17/25 d064cb2f7ffd
tika 18/25 3fb58b7b5dc6
tika 19/25 e759bbbc8afc
tika 20/25 9fce256813d1
tika 21/25 b0a87ad5afe0
tika 22/25 e1da9a1fc19a
tika 23/25 580824e1088a
tika 24/25 f7079fdafc6d
tika 25/25 a8d1e675a2c1
maven: 25 candidate commits
maven 1/25 fba7b5d801ca
maven 2/25 406b0caa7cb6
maven 3/25 1f34a9674295
maven 4/25 4ce99cda4026
maven 5/25 0abaebbe2917
maven 6/25 4dab452c029e
maven 7/25 ca1a69cf349f
maven 8/25 68eea5fd4d98
maven 9/25 03b7d6139d8a
maven 10/25 bdf2c3325042
maven 11/25 a75e6d61dd68
maven 12/25 c5ccb6de8921
maven 13/25 62ec2d0b396b
maven 14/25 688ac68a09d7
maven 15/25 0

((152, 16), (102, 13), (625, 5), (0, 0))

In [47]:
#inspecting the dataset
dataset.head()

,repository,commit,parent_commit,author_date,subject,architecture_smell,metric_name,metric_before,metric_after,metric_delta,improved,refactoring_labels,ranked_refactoring_labels,top_refactoring_label,input_text,target_text
0,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Hub-like Dependency,hub_like_packages,1,1,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
1,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Large Component,large_packages,2,2,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Large Component. Metric large_packages changed from 2 to 2. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
2,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Excessive Package Coupling,package_edges,133,131,2,True,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Excessive Package Coupling. Metric package_edges changed from 133 to 131. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
3,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Cyclic Dependency,cyclic_packages,2,2,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Cyclic Dependency. Metric cyclic_packages changed from 2 to 2. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class
4,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Hub-like Dependency,hub_like_packages,1,1,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class


In [48]:
# inspecting the smells in our dataset
dataset["architecture_smell"].value_counts()

architecture_smell
Excessive Package Coupling    51
Cyclic Dependency             42
Large Component               38
Hub-like Dependency           21
Name: count, dtype: int64

In [50]:
# what are the first 20 top refactoring labels/techniques
dataset["top_refactoring_label"].value_counts().head(20)

top_refactoring_label
Move Class                       21
Change Attribute Type            21
Rename Method                    16
Add Attribute Modifier           11
Change Variable Type             11
Extract Method                    9
Move Method                       8
Extract Interface                 6
Add Thrown Exception Type         5
Rename Class                      5
Change Method Access Modifier     4
Change Return Type                4
Add Method Annotation             4
Remove Thrown Exception Type      3
Encapsulate Attribute             3
Add Parameter Modifier            3
Move Source Folder                3
Change Type Declaration Kind      3
Move And Rename Class             3
Change Parameter Type             3
Name: count, dtype: int64

In [51]:
errors.head(20)

""


In [52]:
#saving final dataset to be used for fine tuning transformer based model for refactoring
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dataset.to_csv(
    OUTPUT_DIR / "architecture_smell_refactoring_dataset.csv",
    index=False,
)

dataset.to_json(
    OUTPUT_DIR / "architecture_smell_refactoring_dataset.jsonl",
    orient="records",
    lines=True,
)

metrics.to_csv(
    OUTPUT_DIR / "before_after_architecture_metrics.csv",
    index=False,
)

ranking.to_csv(
    OUTPUT_DIR / "refactoring_ranking_labels.csv",
    index=False,
)

errors.to_csv(
    OUTPUT_DIR / "mining_errors.csv",
    index=False,
)

repo_summary.to_csv(
    OUTPUT_DIR / "repository_summary.csv",
    index=False,
)

print("Saved to:", OUTPUT_DIR.resolve())

Saved to: C:\dsarp_outputs


In [53]:
# lets get the total number of instances
import json
from pathlib import Path
from collections import Counter
import pandas as pd

dataset_path = Path(r"C:\dsarp_outputs\architecture_smell_refactoring_dataset.jsonl")

rows = []
with dataset_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Total instances:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())
display(df["architecture_smell"].value_counts())
display(df["top_refactoring_label"].value_counts().head(20))
display(df["metric_name"].value_counts())
display(df["improved"].value_counts())

Total instances: 152
Columns: ['repository', 'commit', 'parent_commit', 'author_date', 'subject', 'architecture_smell', 'metric_name', 'metric_before', 'metric_after', 'metric_delta', 'improved', 'refactoring_labels', 'ranked_refactoring_labels', 'top_refactoring_label', 'input_text', 'target_text']


,repository,commit,parent_commit,author_date,subject,architecture_smell,metric_name,metric_before,metric_after,metric_delta,improved,refactoring_labels,ranked_refactoring_labels,top_refactoring_label,input_text,target_text
0,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Hub-like Dependency,hub_like_packages,1,1,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
1,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Large Component,large_packages,2,2,0,False,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Large Component. Metric large_packages changed from 2 to 2. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
2,tika,2b86daf8815f3c8d974f887e4cc064b9159db595,b27dfe0ce70beded9008df90e6a16c2ef66ba9aa,2007-08-17T19:11:25+00:00,TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Excessive Package Coupling,package_edges,133,131,2,True,Remove Thrown Exception Type,Remove Thrown Exception Type,Remove Thrown Exception Type,Architecture smell: Excessive Package Coupling. Metric package_edges changed from 133 to 131. Commit message: TIKA-8: Replaced the jmimeinfo dependency with a trivial mime type detector.,Remove Thrown Exception Type
3,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Cyclic Dependency,cyclic_packages,2,2,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Cyclic Dependency. Metric cyclic_packages changed from 2 to 2. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class
4,tika,f8183f2dffaac516ea48fc3522c302fa4937b26c,033a07cf008dfaf63c2d4bb76bebf184f8b7a338,2007-09-24T16:36:47+00:00,"TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Hub-like Dependency,hub_like_packages,1,1,0,False,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type|Change Parameter Type|Change Thrown Exception Type|Rename Class,Change Attribute Type,"Architecture smell: Hub-like Dependency. Metric hub_like_packages changed from 1 to 1. Commit message: TIKA-17 - Rename all ""Luis"" classes to be ""Tika"" classes",Change Attribute Type | Change Parameter Type | Change Thrown Exception Type | Rename Class


architecture_smell
Excessive Package Coupling    51
Cyclic Dependency             42
Large Component               38
Hub-like Dependency           21
Name: count, dtype: int64

top_refactoring_label
Move Class                       21
Change Attribute Type            21
Rename Method                    16
Add Attribute Modifier           11
Change Variable Type             11
Extract Method                    9
Move Method                       8
Extract Interface                 6
Add Thrown Exception Type         5
Rename Class                      5
Change Method Access Modifier     4
Change Return Type                4
Add Method Annotation             4
Remove Thrown Exception Type      3
Encapsulate Attribute             3
Add Parameter Modifier            3
Move Source Folder                3
Change Type Declaration Kind      3
Move And Rename Class             3
Change Parameter Type             3
Name: count, dtype: int64

metric_name
package_edges        51
cyclic_packages      42
large_packages       38
hub_like_packages    21
Name: count, dtype: int64

improved
False    136
True      16
Name: count, dtype: int64